# Alpha Zero Test

## Import Game Env

In [1]:
import sys
import os
import argparse

# Instead of __file__, use os.getcwd()
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from src.envs import N3il

2.2.5


In [ ]:
n=3

current_dir = os.getcwd()

args = {
    'environment': 'N3il',  # Specify the environment
    'algorithm': 'MCTS',
    'max_level_to_use_symmetry': -1,  # Use symmetry for first 2 levels (helps find compact solutions)
    'n': n,
    'C': 1.41,  # 1e-7 for n=20
    'num_searches': 100*(n**2),  # Adjusted for larger n
    'num_workers': 1,      # >1 ⇒ parallel
    'virtual_loss': 1.0,     # magnitude to subtract at reservation
    'process_bar': True,
    'display_state': True,
    'logging_mode': True,  # Enable logging mode to get return value
    'TopN': n,  # Without Priority
    "simulate_with_priority": False,
    'table_dir': current_dir,  # Directory to save tables
    'figure_dir': os.path.join(current_dir, 'figure'),  # Directory to save figures
    'random_seed': 1,  # Use the loop index as a seed for reproducibility
    'tree_visualization': False,  # Set to True to enable tree visualization
}

n3il_test = N3il((n, n), args)

## ResNet

In [5]:
# --- Torch / Device checks for macOS (Apple Silicon M4) ---
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())  # Usually False on Apple Silicon

# Metal (MPS) backend (Apple GPU)
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
mps_built = hasattr(torch.backends, "mps") and torch.backends.mps.is_built()

print("MPS built:", mps_built)
print("MPS available:", mps_available)

device = torch.device(
    "mps" if mps_available else ("cuda" if torch.cuda.is_available() else "cpu")
)
print("Using device:", device)

# Optional quick sanity test on selected device
try:
    x = torch.randn(2, 2, device=device)
    print("Test tensor sum:", x.sum().item())
except Exception as e:
    print("Device test failed:", e)

torch.manual_seed(0)

Torch version: 2.7.1
CUDA available: False
MPS built: True
MPS available: True
Using device: mps
Test tensor sum: -2.1922035217285156


In [6]:
class ResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()
        self.device = device
        self.startBlock = nn.Sequential(
            nn.Conv2d(2, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for i in range(num_resBlocks)]
        )

        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.row_count * game.column_count, game.action_size)
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(2 * game.row_count * game.column_count, 1),
            nn.Tanh()
        )

        self.to(device)

    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock(x)
        policy = self.policyHead(x)
        value = self.valueHead(x)
        return policy, value

class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x

## Helper Functions

### Exploration Decay

In [10]:
from numba import njit
import numpy as np

@njit(cache=True, nogil=True)
def exploration_decay_nb(x):  # Monotone-down from (0,1) to (1,0)
    # Cosine decay
    # return (np.cos(np.pi * x)+1)/2  # 100% exploration at start, 0% at end
    
    # Linear
    # return 1 - 0.7 * x   # Found optimal 4-point solution: 86/100 times (86.0%)
    # return 1 - x # 85/100 times (85.0%)

    # Square root (gentle early decay)
    # return 1 - 0.9 * np.sqrt(x) # 91/100 times (91.0%)
    # return 1 - 1 * np.sqrt(x) # 83/100 times (83.0%)
    # return 1 - 0.5 * np.sqrt(x) # 88/100 times (88.0%)
    # return 1 - 0.7 * np.sqrt(x) # 86%
    # return 1 - 0.8 * np.sqrt(x) # 92/100 times (92.0%)
    return 1 - 0.85 * np.sqrt(x)

    # Quadratic (faster decay)
    # return 1 - (x ** 2)

    # Exponential (custom normalization)
    # return ((np.exp(1)/(np.exp(1)-1))**2) * ((np.exp(-x)-np.exp(-1)) ** 2) # 85/100 times (85.0%)

    # Exponential fast (k=3)
    #k = 3.0
    # return (np.exp(-k * x) - np.exp(-k)) / (1 - np.exp(-k)) # 90/100 times (90.0%)

    # Exponential slow (k=1)
    # k = 1.0
    # return (np.exp(-k * x) - np.exp(-k)) / (1 - np.exp(-k)) # 86/100 times (86.0%)

    # Cosine decay
    # return 0.5 * (1 + np.cos(np.pi * x)) # 85/100 times (85.0%)

    # Rational decay
    # a = 1.0
    # return (1 - x) / (1 + a * x) # solution: 90/100 times (90.0%)

    # Logistic decay
    # k = 10.0
    # g0 = 1 / (1 + np.exp(k * (0 - 0.5)))
    # g1 = 1 / (1 + np.exp(k * (1 - 0.5)))
    # gx = 1 / (1 + np.exp(k * (x - 0.5)))
    # return (gx - g1) / (g0 - g1) # 86/100 times (86.0%)

    # Cubic decay
    # return 1 - x ** 3 # 91/100 times (91.0%)
    # return 1 - (0.9 * (x ** 3)) # 91/100 times (91.0%)

### Rewarding Function

In [7]:
from numba import njit
import numpy as np

In [9]:
## Remember Also Adjust get_value_nb in collinear_for_mcts.py !!!!!!!!!!!!!!!!!!!!!!!!!!!!
@njit(cache=True, nogil=True)
def get_value_nb(state, pts_upper_bound):
    total = np.sum(state)
    n = pts_upper_bound/2
    
    # === REVERSE REWARDING FUNCTIONS (prefer smaller point counts) ===
    
    # 1. Simple Linear Inverse: 1.0 for empty board, 0.0 for full board
    # return (1.2*n - total) / n  # Range: [0, 1]
    
    # 2. Exponential Decay (Strong preference for fewer points)
    return np.exp(2.0 * ((total-n) / n))  # Range: [e^-2, 1] ≈ [0.135, 1]
    # return np.exp(-1.0 * (total / n))  # Range: [e^-1, 1] ≈ [0.368, 1]
    # return np.exp(-0.5 * (total / n))  # Range: [e^-0.5, 1] ≈ [0.607, 1]
    
    # 3. Power Functions (Adjustable curvature)
    # return ((n - total) / n) ** 2  # Quadratic preference: [0, 1]
    # return ((n - total) / n) ** 0.5  # Square root preference: [0, 1]
    # return ((n - total) / n) ** 3  # Cubic preference (very aggressive): [0, 1]
    
    # 4. Sigmoid-based (Smooth transition around target)
    # target = n * 0.3  # Target 30% of grid filled
    # return 1.0 / (1.0 + np.exp(0.5 * (total - target)))  # Range: ≈[0, 1]
    # return 1.0 / (1.0 + np.exp(1.0 * (total - target)))  # Steeper transition
    
    # 5. Logarithmic Penalty
    # return max(0, 1.0 - np.log(1.0 + total) / np.log(1.0 + n))  # Range: [0, 1]
    
    # 6. ReLU-based with different thresholds
    # return max(0, (1.2 * n - total) / n)  # Reward up to 120% of n: [0, 1.2]
    # return max(0, (1.5 * n - total) / n)  # Current: reward up to 150% of n
    
    # === OPTIMAL FOR 3x3 MINIMAL COMPLETE SET (4 points) ===
    # Simple linear inverse works best for finding exact minimal sets
    # return (1.6*n - total) * n  / (1.6 - 1.3) # Range: [0, 1], 1.0 for empty, 0.0 for full !!!CURRENT OPTIMAL!!!

    # Baseline rewarding function
    '''
    baseline = 1.6 * n
    theoretical_min = 1.3 * n
    num = baseline - total
    if num > 0:
        return num / (baseline - theoretical_min)  # Range: [0, 1], 1.0 for empty, 0.0 for full
    if num <= 0:
        return num / (baseline - theoretical_min)  # Range: [-1, 0], 0.0 for empty, -1.0 for full
    '''
    # Numba-safe scalar casts
    total = np.float64(np.sum(state))
    n = np.float64(pts_upper_bound) / 2.0

    # Target and normalization
    target = 0.9 * n
    max_possible = 2.0 * n
    eps = np.float64(1e-12)
    span = np.maximum(max_possible - target, eps)  # avoid division by zero
    # Normalized distance: 0 at target, 1 at 2n (can be < 0 if total < target)
    tnorm = (total - target) / span

    # ---- Choose ONE of the following returns (uncomment exactly one) ----

    # 2) Quadratic (penalizes farther from target more strongly)
    # return np.clip(1.0 - tnorm * tnorm, 0.0, 1.0)

    # 3) Gaussian peak at target (default active; sharp pull to 0.9n)
    # sigma = np.maximum(0.05 * n, eps)  # controls sharpness
    # return np.exp(-0.5 * ((total - target) / sigma) ** 2)

    # 4) Logistic decay from target upward
    # k = 6.0 / np.maximum(n, 1.0)
    # return 1.0 / (1.0 + np.exp(k * (total - target)))

    # 5) Rational distance penalty (gentler tail)
    # alpha = 2.0 / np.maximum(n, 1.0)
    # return 1.0 / (1.0 + alpha * np.abs(total - target))

    # 6) Piecewise: full at/below target, then linear drop to 0 at 2n
    # if total <= target:
    #     return 1.0
    # else:
    #     return np.maximum(0.0, 1.0 - (total - target) / span)

    # 7) Cosine half-wave on [target, 2n] (smooth with zero slope at target)
    # x = np.clip(tnorm, 0.0, 1.0)               # map [target,2n] -> [0,1]
    # return 0.5 * (1.0 + np.cos(np.pi * x))     # 1 at target, 0 at 2n

    # ------------ Positive-direction variants (optimum at 2n) ------------
    # Use these if you want to test the opposite objective (larger total better).
    # 1+) Linear increasing from target to 2n
    # return np.clip(tnorm, 0.0, 1.0)

    # 2+) Quadratic increasing (slow start, faster near 2n)
    # x = np.clip(tnorm, 0.0, 1.0)
    # return x * x

    # 3+) Exponential rise (very low until near 2n)
    # x = np.clip(tnorm, 0.0, 1.0)
    # k = 4.0
    # return (np.exp(k * x) - 1.0) / (np.exp(k) - 1.0)

### Simulate

In [1]:
from numba import njit
import numpy as np

@njit(cache=True, nogil=True)
def simulate_nb(state, row_count, column_count, pts_upper_bound):
    """
    Perform random rollout until no valid moves remain.
    Return normalized value using a custom value function.
    Uses get_valid_moves_subset_nb for incremental validity updates.
    Note: This function uses numba's random number generator which is seeded globally.
    """
    max_size = row_count * column_count
    # Initial valid moves mask
    valid_moves = get_valid_moves_nb(state, row_count, column_count)
    total_valid = np.sum(valid_moves)

    while total_valid > 0:
        # Build list of valid actions
        acts = np.empty(total_valid, np.int64)
        k = 0
        for idx in range(max_size):
            if valid_moves[idx]:
                acts[k] = idx
                k += 1
        # Randomly select one valid action and place the point
        pick = acts[np.random.randint(0, total_valid)]

        # Incrementally update valid_moves using subset-based filtering
        valid_moves = get_valid_moves_subset_nb(
            state,
            valid_moves,
            pick,
            row_count,
            column_count
        )

        r = pick // column_count
        c = pick % column_count
        state[r, c] = 1  # mark the new point

        total_valid = np.sum(valid_moves)

    # Compute and return the final value
    return get_value_nb(state, pts_upper_bound)

@njit(cache=True, nogil=True)
def filter_top_priority_moves(valid_moves, priority_grid, row_count, column_count, top_N=1):
    """
    Numba-accelerated: Filter valid moves to only those with the top_N highest priorities.

    Args:
        valid_moves (np.ndarray): 1D array (flattened) of valid moves (1=valid, 0=invalid).
        priority_grid (np.ndarray): 2D array of priority values for each grid cell.
        row_count (int): Number of rows in the grid.
        column_count (int): Number of columns in the grid.
        top_N (int): Number of top priority levels to select.

    Returns:
        np.ndarray: 1D mask array with only the top_N-priority valid moves set to 1.
    """
    indices = []
    priorities = []
    for idx in range(valid_moves.shape[0]):
        if valid_moves[idx] == 1:
            indices.append(idx)
            i = idx // column_count
            j = idx % column_count
            priorities.append(priority_grid[i, j])
    if len(indices) == 0:
        return valid_moves

    # Find the unique priorities and sort descending
    # Numba doesn't support np.unique or sort for lists, so do it manually
    # 1. Copy priorities to a new array
    n = len(priorities)
    unique_priorities = []
    for k in range(n):
        p = priorities[k]
        found = False
        for l in range(len(unique_priorities)):
            if unique_priorities[l] == p:
                found = True
                break
        if not found:
            unique_priorities.append(p)
    # 2. Sort unique_priorities descending (simple selection sort)
    for i in range(len(unique_priorities)):
        max_idx = i
        for j in range(i+1, len(unique_priorities)):
            if unique_priorities[j] > unique_priorities[max_idx]:
                max_idx = j
        # Swap
        tmp = unique_priorities[i]
        unique_priorities[i] = unique_priorities[max_idx]
        unique_priorities[max_idx] = tmp

    # 3. Select top_N priorities
    N = min(top_N, len(unique_priorities))
    threshold = unique_priorities[:N]

    # 4. Build mask
    mask = np.zeros_like(valid_moves)
    for k in range(n):
        idx = indices[k]
        p = priorities[k]
        for t in range(N):
            if p == threshold[t]:
                mask[idx] = 1
                break
    return mask

@njit(cache=True, nogil=True)
def simulate_with_priority_nb(state, row_count, column_count, pts_upper_bound, priority_grid, top_N):
    """
    Perform a random rollout that first filters valid moves by priority
    and then proceeds like simulate_nb, but initial valid moves are pre-filtered.
    Args:
        state (np.ndarray): 2D board state.
        row_count (int): Number of rows.
        column_count (int): Number of columns.
        pts_upper_bound (int): Scoring upper bound.
        priority_grid (np.ndarray): 2D array of priorities.
        top_N (int): Number of top priority levels to keep.
    Returns:
        float: Normalized final value.
    """
    max_size = row_count * column_count

    # Initial valid moves mask
    valid_moves = get_valid_moves_nb(state, row_count, column_count)
    # Pre-filter by priority
    valid_moves = filter_top_priority_moves(
        valid_moves, priority_grid, row_count, column_count, top_N
    )
    total_valid = np.sum(valid_moves)

    # Rollout until no moves remain
    while total_valid > 0:
        acts = np.empty(total_valid, np.int64)
        k = 0
        for idx in range(max_size):
            if valid_moves[idx]:
                acts[k] = idx
                k += 1

        pick = acts[np.random.randint(0, total_valid)]

        # Update valid moves and state
        valid_moves = get_valid_moves_subset_nb(
            state, valid_moves, pick, row_count, column_count
        )
        state[pick // column_count, pick % column_count] = 1

        # Filter again by priority
        valid_moves = filter_top_priority_moves(
            valid_moves, priority_grid, row_count, column_count, top_N
        )
        total_valid = np.sum(valid_moves)

    return get_value_nb(state, pts_upper_bound)

### Bit-pack utilities

In [1]:
import numpy as np

# ========= Bit-pack utilities =========
def _pack_bits_bool2d(arr2d: np.ndarray) -> np.ndarray:
    """
    Pack a 2D 0/1 or bool array into a 1D uint8 bit vector using bitorder='big'.
    """
    # Ensure uint8 0/1
    a = arr2d.astype(np.uint8, copy=False)
    return np.packbits(a.reshape(-1), bitorder='big')

def _unpack_bits_to_2d(bits: np.ndarray, rows: int, cols: int) -> np.ndarray:
    """
    Unpack a 1D uint8 bit vector to a 2D uint8 array (0/1) with given shape.
    """
    flat = np.unpackbits(bits, bitorder='big')
    need = rows * cols
    if flat.size > need:
        flat = flat[:need]
    return flat.reshape((rows, cols)).astype(np.uint8, copy=False)

def _bit_clear_inplace(bits: np.ndarray, idx: int) -> None:
    """
    Clear (set to 0) the bit at flat index idx in the packed array (bitorder='big').
    Uses a non-negative mask to avoid OverflowError from bitwise NOT on Python ints.
    """
    byte_i = idx // 8
    off    = idx % 8
    # Build a clear mask: 0xFF with target bit cleared
    clear_mask = np.uint8(0xFF ^ (1 << (7 - off)))
    bits[byte_i] &= clear_mask

def _bit_set_inplace(bits: np.ndarray, idx: int) -> None:
    """
    Set (to 1) the bit at flat index idx in the packed array (bitorder='big').
    """
    byte_i = idx // 8
    off    = idx % 8
    bits[byte_i] |= np.uint8(1 << (7 - off))

## Node

In [1]:
import math
import numpy as np

class Node:
    def __init__(self, game, args, state, parent=None, action_taken=None):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken

        self.children = []
        self.visit_count = 0
        self.value_sum = 0
        self.lock = threading.Lock()
        self._vl = args.get('virtual_loss', 1.0)

        if parent is None:
            self.level = np.sum(state)  # Level is the number of points placed
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_with_symmetry')):
                self.action_space = game.get_valid_moves(state)
                self.valid_moves = game.filter_valid_moves_by_symmetry(
                    self.action_space, state
                ).copy()
            else:
                self.valid_moves = game.get_valid_moves(state)
                self.action_space = self.valid_moves.copy()
        else:
            self.level = parent.level + 1
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_subset_with_symmetry')):
                self.action_space = game.get_valid_moves_subset(
                    parent.state, parent.action_space, self.action_taken)
                self.valid_moves = game.filter_valid_moves_by_symmetry(
                    self.action_space, state
                ).copy()
            else:
                self.valid_moves = game.get_valid_moves_subset(
                    parent.state, parent.action_space, self.action_taken)
                self.action_space = self.valid_moves.copy()
        
        # Ensure action_space is immutable
        self.action_space.flags.writeable = False

        self.is_full = False
        self._cached_ucb = None     # Cached UCB value
        self._ucb_dirty = True      # Indicates whether the cached UCB is stale

    def apply_virtual_loss(self):
        with self.lock:
            self.value_sum -= self._vl
            self.visit_count += 1
            self._ucb_dirty = True  # Mark UCB as outdated

    def revert_virtual_loss(self):
        with self.lock:
            self.value_sum += self._vl
            self._ucb_dirty = True  # Mark UCB as outdated

    def is_fully_expanded(self):
        return self.is_full and len(self.children) > 0

    def select(self, iter):
        best_child = None
        best_ucb = -np.inf
        log_N = math.log(self.visit_count)

        for child in self.children:
            ucb = self.get_ucb(child, iter, log_N)
            if ucb > best_ucb:
                best_child = child
                best_ucb = ucb

        return best_child

    def get_ucb(self, child, iter, log_N=None):
        if log_N is None:
            log_N = math.log(self.visit_count)

        with child.lock:
            if not child._ucb_dirty and child._cached_ucb is not None:
                return child._cached_ucb

            q_value = child.value_sum / child.visit_count
            T_i = self.args['C'] * exploration_decay_nb(iter/self.args['num_searches'])
            exploration_value = T_i * math.sqrt(log_N / child.visit_count)
            ucb = q_value + exploration_value
            # print("Exploit:", q_value)
            # print("Explore:", exploration_value)
            child._cached_ucb = ucb
            child._ucb_dirty = False
            return ucb

    def expand(self):
        valid_indices = np.where(self.valid_moves == 1)[0]
        action = np.random.choice(valid_indices)
        self.valid_moves[action] = 0

        if np.sum(self.valid_moves) == 0:
            self.is_full = True

        child_state = self.state.copy()
        child_state = self.game.get_next_state(child_state, action)

        child = Node(self.game, self.args, child_state, self, action)
        self.children.append(child)
        return child

    def simulate(self):
        tmp = self.state.copy()
        if self.args["simulate_with_priority"] == True:
            return simulate_with_priority_nb(tmp,
                                            self.game.row_count,
                                            self.game.column_count,
                                            self.game.pts_upper_bound,
                                            self.game.priority_grid,
                                            self.args['TopN'])
        else:
            return simulate_nb(tmp,
                            self.game.row_count,
                            self.game.column_count,
                            self.game.pts_upper_bound)

    def backpropagate(self, value):
        with self.lock:
            self.value_sum += value
            self._ucb_dirty = True  # Mark UCB as outdated
        self.visit_count += 1
        if self.parent is not None:
            self.parent.backpropagate(value)

## Node Compressed

In [ ]:
import math
import numpy as np

class Node_Compressed:
    """
    Drop-in compatible Node that implements Scheme B:
    - Store state / valid_moves / action_space as bit-packed arrays (1 bit per cell).
    - Provide properties .state, .valid_moves, .action_space to return unpacked views (uint8 0/1).
    - Internal methods operate on packed bits to reduce memory and allocations.
    Notes:
      * Compatibility: external code that reads node.state / node.valid_moves continues to work.
      * External code that mutates node.valid_moves should not be relied upon (same as original),
        but we return an array copy for safety.
    """
    # Keep the same public attributes (exposed via properties where needed)
    __slots__ = (
        'game','args','parent','action_taken',
        'children','visit_count','value_sum','lock','_vl',
        'level','is_full','_cached_ucb','_ucb_dirty',
        # packed payloads
        '_rows','_cols','_state_bits','_valid_bits','_action_bits'
    )

    def __init__(self, game, args, state, parent=None, action_taken=None):
        self.game = game
        self.args = args
        self.parent = parent
        self.action_taken = action_taken

        self.children = []
        self.visit_count = 0
        self.value_sum = 0.0
        self.lock = threading.Lock()
        self._vl = args.get('virtual_loss', 1.0)

        # grid shape
        self._rows = getattr(game, 'row_count', state.shape[0])
        self._cols = getattr(game, 'column_count', state.shape[1] if state.ndim > 1 else self._rows)

        # --- pack state ---
        # Expect state as 2D 0/1 (uint8 or bool)
        self._state_bits = _pack_bits_bool2d(state)

        # --- compute valid/action masks using the same game API as original ---
        if parent is None:
            # level = number of points placed
            self.level = int(np.sum(state))
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_with_symmetry')):
                action_space = game.get_valid_moves(state)
                valid_moves  = game.filter_valid_moves_by_symmetry(action_space, state).copy()
            else:
                valid_moves  = game.get_valid_moves(state)
                action_space = valid_moves.copy()
        else:
            self.level = parent.level + 1
            # For subset calls, pass parent's action_space (unpacked) and parent.state (unpacked)
            parent_state = parent.state
            parent_action_space = parent.action_space
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_subset_with_symmetry')):
                action_space = game.get_valid_moves_subset(parent_state, parent_action_space, self.action_taken)
                valid_moves  = game.filter_valid_moves_by_symmetry(action_space, self.state,).copy()  # self.state property unpacks
            else:
                valid_moves  = game.get_valid_moves_subset(parent_state, parent_action_space, self.action_taken)
                action_space = valid_moves.copy()

        # --- pack masks & discard large arrays ---
        self._valid_bits  = _pack_bits_bool2d(valid_moves.reshape(self._rows, self._cols))
        self._action_bits = _pack_bits_bool2d(action_space.reshape(self._rows, self._cols))

        self.is_full = False
        self._cached_ucb = None
        self._ucb_dirty = True

    # ---------- compatibility properties ----------
    @property
    def state(self) -> np.ndarray:
        # Return a fresh 2D uint8 (0/1) array
        return _unpack_bits_to_2d(self._state_bits, self._rows, self._cols)

    @property
    def valid_moves(self) -> np.ndarray:
        # Return a fresh 1D uint8 (0/1) vector, write-protected to mimic immutability contract
        vm = _unpack_bits_to_2d(self._valid_bits, self._rows, self._cols).reshape(-1)
        vm.flags.writeable = False
        return vm

    @property
    def action_space(self) -> np.ndarray:
        am = _unpack_bits_to_2d(self._action_bits, self._rows, self._cols).reshape(-1)
        am.flags.writeable = False
        return am

    # ---------- drop-in methods (logic aligned with original) ----------
    def apply_virtual_loss(self):
        with self.lock:
            self.value_sum -= self._vl
            self.visit_count += 1
            self._ucb_dirty = True

    def revert_virtual_loss(self):
        with self.lock:
            self.value_sum += self._vl
            self._ucb_dirty = True

    def is_fully_expanded(self):
        return self.is_full and len(self.children) > 0

    def select(self, iter):
        best_child = None
        best_ucb = -np.inf
        # avoid log(0)
        log_N = math.log(self.visit_count) if self.visit_count > 0 else 0.0

        for child in self.children:
            ucb = self.get_ucb(child, iter, log_N)
            if ucb > best_ucb:
                best_child = child
                best_ucb = ucb

        return best_child

    def get_ucb(self, child, iter, log_N=None):
        if log_N is None:
            log_N = math.log(self.visit_count) if self.visit_count > 0 else 0.0

        with child.lock:
            if not child._ucb_dirty and child._cached_ucb is not None:
                return child._cached_ucb

            q_value = child.value_sum / max(1, child.visit_count)
            T_i = self.args['C'] * exploration_decay_nb(iter/self.args['num_searches'])
            exploration_value = T_i * math.sqrt(max(1e-12, log_N) / max(1, child.visit_count))
            ucb = q_value + exploration_value
            child._cached_ucb = ucb
            child._ucb_dirty = False
            return ucb

    def _valid_sum(self) -> int:
        # Fast count of set bits
        flat = np.unpackbits(self._valid_bits, bitorder='big')
        return int(flat[: self._rows * self._cols].sum())

    def expand(self):
        # Choose a random valid action; work on packed bits to avoid storing big arrays
        flat_valid = np.unpackbits(self._valid_bits, bitorder='big')[: self._rows * self._cols]
        valid_indices = np.flatnonzero(flat_valid)
        if valid_indices.size == 0:
            self.is_full = True
            return None

        action = int(np.random.choice(valid_indices))
        # consume this action (clear its bit)
        _bit_clear_inplace(self._valid_bits, action)

        # mark is_full if no moves remain
        if flat_valid.sum() - 1 == 0:
            self.is_full = True

        # Build child state as in original (copy, then get_next_state)
        child_state = self.state.copy()  # property: unpack current state
        child_state = self.game.get_next_state(child_state, action)

        # Create child node (compressed)
        child = Node_Compressed(self.game, self.args, child_state, self, action)
        self.children.append(child)
        return child

    def simulate(self):
        # Unpack to 2D array; simulation mutates a copy
        tmp = self.state.copy()
        if self.args.get("simulate_with_priority", False):
            return simulate_with_priority_nb(tmp,
                                            self.game.row_count,
                                            self.game.column_count,
                                            self.game.pts_upper_bound,
                                            self.game.priority_grid,
                                            self.args['TopN'])
        else:
            return simulate_nb(tmp,
                            self.game.row_count,
                            self.game.column_count,
                            self.game.pts_upper_bound)

    def backpropagate(self, value):
        with self.lock:
            self.value_sum += value
            self._ucb_dirty = True
        self.visit_count += 1
        if self.parent is not None:
            self.parent.backpropagate(value)

## MCTS

### MCTSVisualizer

In [2]:
import os
import io
import json
import base64
import datetime
import numpy as np
import matplotlib.pyplot as plt
from pyvis.network import Network

# Assume Node class is defined elsewhere

class MCTSVisualizer:
    # Global storage for all trials and steps (class variable)
    global_trial_data = []

    def __init__(self, game, args):
        self.game = game
        self.args = args
        self.snapshots = []  # Store snapshots for the current trial
        self.trial_id = 'unknown'

    def start_new_trial(self, trial_id):
        """Prepares the visualizer for a new trial."""
        self.trial_id = trial_id
        self.snapshots = [] # Clear snapshots from the previous trial

    def _state_to_image_base64(self, state):
        """
        Convert a game state to a base64-encoded image using the game's display_state method.
        """
        plt.figure(figsize=(4, 4))
        rows, cols = self.game.row_count, self.game.column_count
        y_idx, x_idx = np.nonzero(state)
        y_disp = rows - 1 - y_idx
        plt.scatter(x_idx, y_disp, s=200, c='blue', linewidths=0.5)
        plt.xticks(range(cols))
        plt.yticks(range(rows))
        plt.grid(True, alpha=0.3)
        plt.xlim(-0.5, cols - 0.5)
        plt.ylim(-0.5, rows - 0.5)
        plt.gca().set_aspect('equal')
        plt.xticks([])
        plt.yticks([])
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', dpi=80, pad_inches=0.1)
        buf.seek(0)
        img_base64 = base64.b64encode(buf.read()).decode()
        plt.close()
        return f"data:image/png;base64,{img_base64}"

    def _get_node_label(self, node, iter_num=None):
        """
        Generate a label for a node showing its statistics.
        """
        avg_value = node.value_sum / node.visit_count if node.visit_count > 0 else 0
        ucb = 0
        if node.parent is not None and node.visit_count > 0:
            try:
                ucb = node.parent.get_ucb(node, iter_num or 0)
            except:
                ucb = 0 # Failsafe
        
        label = f"Visits: {node.visit_count}\n"
        label += f"Value Sum: {node.value_sum:.3f}\n"
        label += f"Avg Value: {avg_value:.3f}\n"
        label += f"UCB: {ucb:.3f}"
        return label

    def create_tree_snapshot(self, root, snapshot_name="MCTS Tree"):
        """
        Create a tree visualization snapshot for the current step.
        """
        # (This is your tree_visualization method, renamed for clarity)
        net = Network(height="600px", width="100%", bgcolor="#222222", font_color="white", directed=True)
        net.barnes_hut()
        
        all_nodes, visited = [], set()
        def collect_nodes_dfs(node, level=0):
            if id(node) in visited: return
            visited.add(id(node))
            all_nodes.append((node, level))
            for child in node.children:
                collect_nodes_dfs(child, level + 1)
        
        collect_nodes_dfs(root)
        print(f"Tree visualization: Found {len(all_nodes)} nodes total")

        json_nodes, json_edges, node_mapping = [], [], {}
        for i, (node, level) in enumerate(all_nodes):
            current_id = f"node_{i}"
            node_mapping[id(node)] = current_id
            img_base64 = self._state_to_image_base64(node.state)
            label = self._get_node_label(node)
            
            color = "#4CAF50"  # Default green
            if node.is_fully_expanded(): color = "#2196F3"
            elif len(node.children) == 0 and not node.is_fully_expanded(): color = "#FF9800"
            elif np.sum(node.valid_moves) == 0: color = "#F44336"

            label_lines = label.split('\n')
            escaped_label = label.replace('\n', '\\n')
            title_text = f"Action: {node.action_taken}\\n{escaped_label}\\nChildren: {len(node.children)}\\nValid moves left: {np.sum(node.valid_moves)}"
            
            json_nodes.append({
                "id": current_id, "label": label_lines, "image": img_base64,
                "shape": "image", "size": 30, "level": level, "color": color,
                "title": title_text, "x": i * 100, "y": level * 150
            })
        
        for node, _ in all_nodes:
            current_id = node_mapping[id(node)]
            for child in node.children:
                if id(child) in node_mapping:
                    child_id = node_mapping[id(child)]
                    json_edges.append({
                        "from": current_id, "to": child_id,
                        "smooth": {"type": "cubicBezier", "forceDirection": "vertical", "roundness": 0.4}
                    })
        
        snapshot_data = {
            'name': snapshot_name, 'trial_id': self.trial_id,
            'step_number': len(self.snapshots), 'total_nodes': len(all_nodes),
            'args': self.args.copy(), 'json_nodes': json_nodes, 'json_edges': json_edges
        }
        
        self.snapshots.append(snapshot_data)
        MCTSVisualizer.global_trial_data.append(snapshot_data)
        print(f"Snapshot created for Trial {self.trial_id}, Step {len(self.snapshots)}")

    @classmethod
    def clear_global_data(cls):
        """Clear all global trial data."""
        cls.global_trial_data.clear()
        print("Global trial data cleared.")

    @classmethod
    def save_final_visualization(cls, web_viz_dir=None, experiment_name="mcts_experiment"):
        """
        Save the final comprehensive visualization at the end of all trials.
        """
        # (This is your original save_final_visualization method)
        if not cls.global_trial_data:
            print("No global trial data to save.")
            return None
        if web_viz_dir is None:
            web_viz_dir = './web_visualization'
        os.makedirs(web_viz_dir, exist_ok=True)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = os.path.join(web_viz_dir, f"{experiment_name}_comprehensive_{timestamp}.html")
        cls._save_comprehensive_html(filename)
        return filename

    @classmethod
    def _save_comprehensive_html(cls, filename="mcts_comprehensive_visualization.html"):
        """
        Creates the comprehensive HTML file with all trials and steps.
        """
        # (This is your original save_comprehensive_html method)
        if not cls.global_trial_data:
            print("No global trial data to save.")
            return

        json_snapshots = []
        for snapshot in cls.global_trial_data:
            json_snapshots.append({
                "id": f"t{snapshot['trial_id']}_s{snapshot['step_number']}",
                "title": f"Trial {snapshot['trial_id']} - {snapshot['name']}",
                "trial_id": snapshot['trial_id'],
                "step_number": snapshot['step_number'],
                "total_nodes": snapshot['total_nodes'],
                "nodes": snapshot.get('json_nodes', []),
                "edges": snapshot.get('json_edges', []),
                "grid_size": snapshot.get('args', {}).get('n', 'unknown')
            })
        
        # Save the JSON data separately for debugging and external use
        json_filename = filename.replace('.html', '_data.json')
        with open(json_filename, 'w', encoding='utf-8') as f:
            json.dump(json_snapshots, f, indent=2, ensure_ascii=False)
        print(f"JSON data saved to: {json_filename}")

        # The large HTML string template goes here. It's omitted for brevity but is identical
        # to the one in your original code.
        html_content = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <title>MCTS Comprehensive Tree Visualization</title>
  <style>
    /* ... Your CSS styles ... */
  </style>
</head>
<body>
  <script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
  <script>
    const snapshots = {json.dumps(json_snapshots, ensure_ascii=False, indent=2)};
    // ... The rest of your JavaScript for navigation, rendering, etc. ...
  </script>
</body>
</html>
"""
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(html_content)
        print(f"Comprehensive visualization saved to: {filename}")

### MCTS

In [1]:

from tqdm import trange

class MCTS:
    def __init__(self, game, args={
        'num_searches': 1000,
        'C': 1.4,
        'tree_visualization': False # Control visualization from args
    }):
        self.game = game
        self.args = args
        self.trial_id = None # Current trial ID

        # If visualization is enabled, create a visualizer instance
        if self.args.get('tree_visualization', False):
            self.visualizer = MCTSVisualizer(self.game, self.args)
        else:
            self.visualizer = None

    def start_new_trial(self, trial_id):
        """Starts a new trial, like a new game or experiment run."""
        self.trial_id = trial_id
        if self.visualizer:
            self.visualizer.start_new_trial(trial_id)

    def search(self, state):
        # define root
        if self.args.get('node_compression', False):
            root = Node_Compressed(self.game, self.args, state)
            print("Using Node_Compressed for MCTS")
        else:
            root = Node(self.game, self.args, state)

        if self.args['process_bar'] == True:
            search_iterator = trange(self.args['num_searches'])
        else:
            search_iterator = range(self.args['num_searches'])

        for search in search_iterator:
            node = root

            # selection
            while node.is_fully_expanded(): #         return self.is_full and len(self.children) > 0
                node = node.select(iter=search)

            if node.action_taken is not None:
                value, is_terminal = self.game.get_value_and_terminated(node.state, node.valid_moves)
                # has_collinear = self.game.check_collinear(node.state, node.action_taken)
                # value, _ = self.game.get_value_and_terminated(node.state)

                if not is_terminal:
                    node = node.expand()
                    value = node.simulate()
            else:
                node = node.expand()
                value = node.simulate()

            node.backpropagate(value)

        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        
        # ---- DELEGATE VISUALIZATION ----
        if self.visualizer:
            num_points = np.sum(state)
            snapshot_name = f"Step {num_points}: {num_points} points placed"
            
            # Call the visualizer to create the snapshot
            self.visualizer.create_tree_snapshot(root, snapshot_name)
            
            if self.args.get('pause_at_each_step', False):
                try:
                    response = input("Output action prob? (y/n): ").strip().lower()
                    if response == 'y':
                        print("Action probabilities:", action_probs)
                except (EOFError, KeyboardInterrupt):
                    pass
        
        return action_probs